# Cross-step tx→tx graph — Step 3: causal inductive GNN

Step 2's hand-engineered pair features lifted F1 by +0.0025 over the trajectory baseline (0.8098 → 0.8122). The hypothesis we're testing here is whether a learned GNN can extract additional signal that the per-tx aggregations miss — specifically (i) multi-hop propagation along the cross-step graph, (ii) source-feature × Δt × role × source-label interactions, (iii) conditional propagation based on source features.

## Architecture

Input node features (per tx): standardised 108 intrinsic per-tx features. **No label feature on the node itself** — that would leak a tx's own label into its prediction.

Edge attributes (per cross-step edge `T_i → T_j`):
- `dt_norm`: Δt / 49
- `role`: one-hot of {out→in, out→out, in→in, in→out} (4 dims)
- `src_label`: one-hot of {illicit, licit, unknown} (3 dims) — encodes the source's label *into the message*, not the target's input. At inference time, sources are at `t' < t` so their labels reflect the realistic fraud-team-knowledge contract.

Total edge attr dim: 8.

Each layer is a SAGE-style message-passing block:
```
  m_{i→j} = MLP_msg([h_i || edge_attr_{ij}])
  agg_j   = sum_i m_{i→j}      (scatter via index_add_)
  h'_j    = ReLU(LayerNorm( W_self h_j + W_agg agg_j ))
```
Two stacked layers, then a 2-layer MLP head produces the logit.

**Inductive**: no per-node embedding lookups; everything is feature-derived. **Causal**: edges all have `t(src) < t(dst)` strictly (Step 1 verified). A k-layer GNN at tx T sees only its k-hop temporal predecessors, all at `t' < t(T)`.

## Training

- Train split: labelled txs with `t ≤ 29`
- Temporal val (early stopping + threshold tuning): `30 ≤ t ≤ 34`
- Test: `t ≥ 35`

Full-batch forward over the whole graph (203K nodes, 445K edges) on MPS (Apple Metal). Loss: `BCEWithLogitsLoss` with `pos_weight = 1/π_train − 1` to balance illicit class. Adam, lr=1e-3. Early stop on val F1 with patience 10.

## Reference results (same temporal split, this repo)

| | F1 | AUC | PR-AUC |
|---|---|---|---|
| A0 — RF[108 intrinsic] | 0.8021 | 0.9026 | 0.7855 |
| A1 — RF[108 + 103 traj] | 0.8098 | 0.9196 | 0.8029 |
| A3 — RF[A1 + Block B pair feats] | 0.8122 | 0.9160 | 0.8027 |

Targets: GNN-only ≥ 0.80 (don't underperform RF baseline), GNN-stacked ≥ 0.815.

In [1]:
import time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
print(f"repo root: {ROOT}")

TRAIN_END     = 34
VAL_START     = 30   # temporal val: t in [30, 34]
N_TIME_STEPS  = 49
MAX_WALLET_DEGREE = 50
RANDOM_SEED   = 175
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"device: {DEVICE}")

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project
device: mps


In [2]:
print("Loading data...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1, 2:0, 3:-1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + ["in_txs_degree","out_txs_degree"]
feat_cols = [c for c in all_cols if c not in ("txId","Time step") and c not in GRAPH_COLS]
F_INTRINSIC = len(feat_cols)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
X_intr   = tx_features_df[feat_cols].fillna(0).astype(np.float32).values

addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a:i for i,a in enumerate(wallet_addrs)}
addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
wallet_in_txs  = defaultdict(list)
wallet_out_txs = defaultdict(list)
for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
    wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
    wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r:r[1])
for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r:r[1])
all_incidence = defaultdict(set)
for w, lst in wallet_in_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)
for w, lst in wallet_out_txs.items():
    for tx, _ in lst: all_incidence[w].add(tx)

print(f"  N_tx={N_tx:,}  F_INTRINSIC={F_INTRINSIC}")

Loading data...


  N_tx=203,769  F_INTRINSIC=108


In [3]:
print(f"Building cross-step edges (cap={MAX_WALLET_DEGREE})...")
ROLE_OUT_IN, ROLE_OUT_OUT, ROLE_IN_IN, ROLE_IN_OUT = 0, 1, 2, 3
src, dst, dt_arr, role_arr = [], [], [], []
for w in all_incidence:
    if len(all_incidence[w]) < 2 or len(all_incidence[w]) > MAX_WALLET_DEGREE: continue
    events = []
    for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx, 'in'))
    for tx, t in wallet_out_txs.get(w, []): events.append((t, tx, 'out'))
    events.sort(key=lambda r: r[0])
    K = len(events)
    for i in range(K):
        ti, txi, si = events[i]
        for j in range(i+1, K):
            tj, txj, sj = events[j]
            if tj <= ti or txi == txj: continue
            if   si=='out' and sj=='in':  role = ROLE_OUT_IN
            elif si=='out' and sj=='out': role = ROLE_OUT_OUT
            elif si=='in'  and sj=='in':  role = ROLE_IN_IN
            else:                          role = ROLE_IN_OUT
            src.append(txi); dst.append(txj)
            dt_arr.append(tj-ti); role_arr.append(role)
src      = np.array(src, dtype=np.int64)
dst      = np.array(dst, dtype=np.int64)
dt_edges = np.array(dt_arr, dtype=np.int32)
role_e   = np.array(role_arr, dtype=np.int8)
E = len(src)
print(f"  E={E:,}")

# Build edge_attr: [E, 8] = dt_norm + role_onehot(4) + src_label_onehot(3)
dt_norm  = (dt_edges.astype(np.float32) / float(N_TIME_STEPS)).reshape(-1, 1)
role_oh  = np.zeros((E, 4), dtype=np.float32); role_oh[np.arange(E), role_e] = 1.0
src_lab  = tx_label[src]
lab_oh   = np.zeros((E, 3), dtype=np.float32)
lab_oh[src_lab == 1, 0] = 1.0
lab_oh[src_lab == 0, 1] = 1.0
lab_oh[src_lab == -1, 2] = 1.0
edge_attr = np.concatenate([dt_norm, role_oh, lab_oh], axis=1)
F_EDGE = edge_attr.shape[1]
print(f"  edge_attr shape: {edge_attr.shape}")

Building cross-step edges (cap=50)...


  E=445,180
  edge_attr shape: (445180, 8)


In [4]:
labelled   = (tx_label != -1)
train_mask_np = labelled & (tx_time <= 29)              # strictly t<=29 for the GNN train
val_mask_np   = labelled & (tx_time >= VAL_START) & (tx_time <= TRAIN_END)
test_mask_np  = labelled & (tx_time > TRAIN_END)
print(f"train: n={int(train_mask_np.sum()):,}  illicit_rate={tx_label[train_mask_np].mean():.4f}")
print(f"val:   n={int(val_mask_np.sum()):,}  illicit_rate={tx_label[val_mask_np].mean():.4f}")
print(f"test:  n={int(test_mask_np.sum()):,}  illicit_rate={tx_label[test_mask_np].mean():.4f}")

# StandardScaler fit on TRAIN ONLY (t<=29)
scaler = StandardScaler()
scaler.fit(X_intr[train_mask_np])
X_scaled = scaler.transform(X_intr).astype(np.float32)

# Move to torch tensors
X_t       = torch.from_numpy(X_scaled).to(DEVICE)
edge_idx  = torch.from_numpy(np.stack([src, dst], axis=0)).to(DEVICE)   # [2, E]
edge_attr_t = torch.from_numpy(edge_attr).to(DEVICE)
y_full    = torch.from_numpy(tx_label.astype(np.float32)).to(DEVICE)

train_mask = torch.from_numpy(train_mask_np).to(DEVICE)
val_mask   = torch.from_numpy(val_mask_np).to(DEVICE)
test_mask  = torch.from_numpy(test_mask_np).to(DEVICE)

pi_train = float(tx_label[train_mask_np].mean())
pos_weight = torch.tensor([(1.0 - pi_train) / pi_train], device=DEVICE)
print(f"  pos_weight: {pos_weight.item():.2f}")
print(f"  X tensor: {X_t.shape} dtype={X_t.dtype}")
print(f"  edge_idx: {edge_idx.shape}  edge_attr: {edge_attr_t.shape}")

train: n=26,381  illicit_rate=0.1088
val:   n=3,513  illicit_rate=0.1682
test:  n=16,670  illicit_rate=0.0650


  pos_weight: 8.19
  X tensor: torch.Size([203769, 108]) dtype=torch.float32
  edge_idx: torch.Size([2, 445180])  edge_attr: torch.Size([445180, 8])


In [5]:
class CausalSAGELayer(nn.Module):
    """One causal message-passing layer over directed temporal edges (src -> dst).

    Aggregates messages into dst nodes. src nodes are unchanged by aggregation
    (no symmetric edge added) — this is what enforces the temporal flow.
    """
    def __init__(self, d_in, d_edge, d_out, dropout=0.2):
        super().__init__()
        self.msg = nn.Sequential(
            nn.Linear(d_in + d_edge, d_out),
            nn.ReLU(),
            nn.Linear(d_out, d_out),
        )
        self.upd_self = nn.Linear(d_in, d_out)
        self.upd_agg  = nn.Linear(d_out, d_out)
        self.norm     = nn.LayerNorm(d_out)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, h, edge_idx, edge_attr):
        # edge_idx[0]=src, edge_idx[1]=dst
        src_idx = edge_idx[0]
        dst_idx = edge_idx[1]
        h_src = h.index_select(0, src_idx)                        # [E, d_in]
        m = self.msg(torch.cat([h_src, edge_attr], dim=1))        # [E, d_out]
        # scatter sum into dst rows
        agg = torch.zeros(h.size(0), m.size(1), device=h.device, dtype=m.dtype)
        agg.index_add_(0, dst_idx, m)
        out = self.upd_self(h) + self.upd_agg(agg)
        out = self.norm(out)
        out = F.relu(out)
        out = self.dropout(out)
        return out

class CausalGNN(nn.Module):
    def __init__(self, d_in, d_edge, d_hidden=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_hidden)
        self.layers  = nn.ModuleList([
            CausalSAGELayer(d_hidden, d_edge, d_hidden, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.Linear(d_hidden, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, X, edge_idx, edge_attr):
        h = F.relu(self.in_proj(X))
        for layer in self.layers:
            h = layer(h, edge_idx, edge_attr)
        logit = self.head(h).squeeze(-1)
        return logit, h

torch.manual_seed(RANDOM_SEED)
model = CausalGNN(d_in=F_INTRINSIC, d_edge=F_EDGE, d_hidden=64, n_layers=2, dropout=0.2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"  model params: {n_params:,}")

  model params: 45,761


In [6]:
from sklearn.metrics import precision_recall_curve

def eval_split(logits_np, mask_np, y_np):
    p = 1.0 / (1.0 + np.exp(-logits_np[mask_np]))
    y = y_np[mask_np]
    return p, y

def best_thresh_on_val(p_val, y_val):
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    if len(thr) == 0: return 0.5, float(f1s.max())
    bi = int(np.argmax(f1s[:-1])) if len(f1s) > 1 else 0
    return float(thr[bi]), float(f1s[bi])

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
MAX_EPOCHS = 80; PATIENCE = 10
best_val_f1 = -1.0; best_state = None; best_thresh = 0.5; bad = 0
y_full_np = tx_label.astype(np.int64)

print("Training…")
t0 = time.time()
for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    opt.zero_grad()
    logits, _ = model(X_t, edge_idx, edge_attr_t)
    loss = loss_fn(logits[train_mask], y_full[train_mask])
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        logits_eval, _ = model(X_t, edge_idx, edge_attr_t)
        logits_np = logits_eval.detach().cpu().numpy()
    p_val, y_val = eval_split(logits_np, val_mask_np, y_full_np)
    thr, f1_val_at_best = best_thresh_on_val(p_val, y_val)
    f1_val_05 = f1_score(y_val, (p_val >= 0.5).astype(int), pos_label=1, zero_division=0)
    auc_val   = roc_auc_score(y_val, p_val) if y_val.sum() > 0 else float("nan")
    if f1_val_at_best > best_val_f1:
        best_val_f1 = f1_val_at_best
        best_thresh = thr
        best_state  = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch == 1 or epoch % 5 == 0 or bad == 0:
        print(f"  ep{epoch:>3d}  loss={loss.item():.4f}  "
              f"val_F1@0.5={f1_val_05:.4f}  val_F1@best={f1_val_at_best:.4f}  "
              f"thr={thr:.3f}  val_AUC={auc_val:.4f}  bad={bad}")
    if bad >= PATIENCE:
        print(f"  early stop at epoch {epoch}")
        break
print(f"  training done in {time.time()-t0:.0f}s")
print(f"  best val F1 (tuned threshold): {best_val_f1:.4f}  thr={best_thresh:.3f}")
model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

Training…


  ep  1  loss=1.2311  val_F1@0.5=0.3840  val_F1@best=0.6028  thr=0.538  val_AUC=0.7643  bad=0


  ep  2  loss=1.1738  val_F1@0.5=0.3813  val_F1@best=0.6584  thr=0.554  val_AUC=0.7895  bad=0


  ep  3  loss=1.1283  val_F1@0.5=0.3800  val_F1@best=0.6680  thr=0.578  val_AUC=0.8007  bad=0


  ep  4  loss=1.0862  val_F1@0.5=0.3819  val_F1@best=0.6712  thr=0.600  val_AUC=0.8078  bad=0


...

  ep 52  loss=0.2573  val_F1@0.5=0.8046  val_F1@best=0.8921  thr=0.911  val_AUC=0.9790  bad=0


  ep 55  loss=0.3030  val_F1@0.5=0.6753  val_F1@best=0.8501  thr=0.959  val_AUC=0.9695  bad=3


  ep 60  loss=0.2822  val_F1@0.5=0.7650  val_F1@best=0.8335  thr=0.976  val_AUC=0.9713  bad=8


  early stop at epoch 62
  training done in 46s
  best val F1 (tuned threshold): 0.8921  thr=0.911


<All keys matched successfully>

In [7]:
model.eval()
with torch.no_grad():
    logits, h_final = model(X_t, edge_idx, edge_attr_t)
    logits_np = logits.detach().cpu().numpy()
    h_np      = h_final.detach().cpu().numpy()
p_full = 1.0/(1.0+np.exp(-logits_np))

# Test predictions at tuned threshold AND at 0.5
p_test = p_full[test_mask_np]
y_test = y_full_np[test_mask_np]
for thr_name, thr in [("0.5", 0.5), (f"val-tuned {best_thresh:.3f}", best_thresh)]:
    yp = (p_test >= thr).astype(int)
    f1 = f1_score(y_test, yp, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_test, p_test)
    pr  = average_precision_score(y_test, p_test)
    print(f"\n=== GNN-only @ thr={thr_name} ===")
    print(f"  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={pr:.4f}")
    print(classification_report(y_test, yp, target_names=["licit","illicit"], digits=4, zero_division=0))

# Per-timestep at val-tuned threshold
print("\nPer-timestep test F1 (val-tuned threshold):")
test_t_arr = tx_time[test_mask_np]
yp_test_thr = (p_test >= best_thresh).astype(int)
for t in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        f1t = float("nan")
    else:
        f1t = f1_score(yt, yp_test_thr[m], pos_label=1, zero_division=0)
    print(f"  t={t:>2d}  n={int(m.sum()):>4d}  illicit={int(yt.sum()):>3d}  F1={f1t:.4f}")


=== GNN-only @ thr=0.5 ===
  F1=0.3633  AUC=0.9111  PR-AUC=0.7251
              precision    recall  f1-score   support

       licit     0.9829    0.8206    0.8945     15587
     illicit     0.2354    0.7950    0.3633      1083

    accuracy                         0.8190     16670
   macro avg     0.6092    0.8078    0.6289     16670
weighted avg     0.9344    0.8190    0.8600     16670


=== GNN-only @ thr=val-tuned 0.911 ===
  F1=0.5642  AUC=0.9111  PR-AUC=0.7251
              precision    recall  f1-score   support

       licit     0.9806    0.9400    0.9599     15587
     illicit     0.4589    0.7322    0.5642      1083

    accuracy                         0.9265     16670
   macro avg     0.7198    0.8361    0.7620     16670
weighted avg     0.9467    0.9265    0.9342     16670


Per-timestep test F1 (val-tuned threshold):
  t=35  n=1341  illicit=182  F1=0.8825
  t=36  n=1708  illicit= 33  F1=0.4882
  t=37  n= 498  illicit= 40  F1=0.6479
  t=38  n= 756  illicit=111  F1=0.8362

In [8]:
# Stacking (D6): RF on [108 intrinsic + GNN_prob + GNN_emb_8].
# We project the 64-dim hidden to 8 PCA components for a compact graph signal.
from sklearn.decomposition import PCA

pca = PCA(n_components=8, random_state=RANDOM_SEED)
pca.fit(h_np[train_mask_np])
h_pca = pca.transform(h_np).astype(np.float32)
print(f"  GNN hidden PCA explained var: {pca.explained_variance_ratio_.sum():.3f}")

# Build augmented feature matrix
X_aug = np.concatenate([X_intr, p_full.astype(np.float32).reshape(-1,1), h_pca], axis=1)
print(f"  X_aug shape: {X_aug.shape}")

# Need traj features (103) for stacking against A1 baseline. Quick recompute or skip.
# For simplicity, the stacking baseline here uses only intrinsic + GNN signal.
# A separate cell can add traj features if desired.

# Use the labelled-train set (t <= TRAIN_END) for the stacker, since the GNN's
# train-only output was generated for ALL nodes. Stacker training mask = labelled & t <= 34.
stk_train_mask = (tx_label != -1) & (tx_time <= TRAIN_END)
stk_test_mask  = test_mask_np
y_stk_train = tx_label[stk_train_mask]
y_stk_test  = tx_label[stk_test_mask]

rf_stk = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 n_jobs=-1, random_state=RANDOM_SEED)
rf_stk.fit(X_aug[stk_train_mask], y_stk_train)
yp = rf_stk.predict(X_aug[stk_test_mask])
ypr= rf_stk.predict_proba(X_aug[stk_test_mask])[:,1]
f1 = f1_score(y_stk_test, yp, pos_label=1, zero_division=0)
auc= roc_auc_score(y_stk_test, ypr)
pr = average_precision_score(y_stk_test, ypr)
print(f"\n=== Stacked: RF[108 intrinsic + GNN prob + GNN PCA8] ===")
print(f"  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={pr:.4f}")
print(classification_report(y_stk_test, yp, target_names=["licit","illicit"], digits=4, zero_division=0))

  GNN hidden PCA explained var: 0.850
  X_aug shape: (203769, 117)



=== Stacked: RF[108 intrinsic + GNN prob + GNN PCA8] ===
  F1=0.7931  AUC=0.9070  PR-AUC=0.7892
              precision    recall  f1-score   support

       licit     0.9810    0.9929    0.9870     15587
     illicit     0.8770    0.7239    0.7931      1083

    accuracy                         0.9755     16670
   macro avg     0.9290    0.8584    0.8900     16670
weighted avg     0.9743    0.9755    0.9744     16670



In [9]:
print("=" * 75)
print("SUMMARY (test t >= 35)")
print("=" * 75)
print(f"  Reference baselines (existing notebooks):")
print(f"    A0 — RF[108 intrinsic]                F1=0.8021  AUC=0.9026  PR-AUC=0.7855")
print(f"    A1 — RF[108 + 103 traj]               F1=0.8098  AUC=0.9196  PR-AUC=0.8029")
print(f"    A3 — RF[A1 + Block B pair feats]      F1=0.8122  AUC=0.9160  PR-AUC=0.8027")
print()
print(f"  GNN-only @ val-tuned thr={best_thresh:.3f}:    "
      f"F1={f1_score(y_test, (p_test>=best_thresh).astype(int), zero_division=0):.4f}  "
      f"AUC={roc_auc_score(y_test, p_test):.4f}  "
      f"PR-AUC={average_precision_score(y_test, p_test):.4f}")
print(f"  GNN stacked into RF[108 + prob + emb]:  F1={f1:.4f}  AUC={auc:.4f}  PR-AUC={pr:.4f}")

SUMMARY (test t >= 35)
  Reference baselines (existing notebooks):
    A0 — RF[108 intrinsic]                F1=0.8021  AUC=0.9026  PR-AUC=0.7855
    A1 — RF[108 + 103 traj]               F1=0.8098  AUC=0.9196  PR-AUC=0.8029
    A3 — RF[A1 + Block B pair feats]      F1=0.8122  AUC=0.9160  PR-AUC=0.8027

  GNN-only @ val-tuned thr=0.911:    F1=0.5642  AUC=0.9111  PR-AUC=0.7251
  GNN stacked into RF[108 + prob + emb]:  F1=0.7931  AUC=0.9070  PR-AUC=0.7892
